# Tổng quan Milestone 2: Phân tích cấu trúc phân cấp, chuẩn hóa và khử trùng

## 1. Thực thi đồng nhất trên 25.000 bài báo bằng Google Colab CPU-only

Toàn bộ 25.000 bài báo (41.498 phiên bản) của 5 thành viên được phân tích mới hoàn toàn từ đầu (`publications_reused = 0`, cùng phiên bản mã nguồn `code_hash = cf2de7a143c8...`) thông qua 5 phiên Google Colab Standard CPU-only độc lập, sau đó hợp nhất và kiểm chuẩn toàn cục 5 bảng dữ liệu bằng [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="dataframe ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
draw_pipeline_diagram(
    "Hình M2.0. Quy trình phân tích cấu trúc LaTeX, chuẩn hóa và khử trùng 2 tầng của Milestone 2",
    [
        ("1. Gom tệp nguồn", "Xác định tệp gốc biên dịch\nĐối sánh tiêu đề M1 (1.706 bản)\nMở rộng \\input và \\include"),
        ("2. Dựng cây phân cấp", "Document, Section, List\nSentence, Formula, Figure\nLoại References, giữ Appendix"),
        ("3. Chuẩn hóa và BibTeX", "Đồng nhất ký hiệu toán học\nHợp nhất .bib, \\bibitem, .bbl\nĐổi khóa \\cite{} xuyên bản"),
        ("4. Khử trùng và kiểm chuẩn", "Kho elements SHA-256 (-35,04%)\nHợp nhất bibtex (-35,83%)\nKiểm chuẩn SQLite DAG 0 lỗi"),
    ],
)

In [ ]:
tex_total = m2["versions_with_tex"]
labels = [
    ("success", "Thành công hoàn toàn (success)", True),
    ("partial", "Phân tích bán phần (partial, giữ cây hợp lệ)", True),
    ("empty_no_tex", "Không có TeX (chỉ có PDF hoặc thiếu .tex)", False),
    ("error", "Lỗi cú pháp TeX nặng hoặc thiếu tệp gốc (error)", True),
    ("error_missing_source", "Thiếu nguồn từ M1 (169 HTTP 404 và 11 bản chờ)", False),
]
rows_st = []
for k, lbl, in_tex in labels:
    cnt = int(m2["status_counts"].get(k, 0))
    rows_st.append({
        "Trạng thái phân tích": lbl,
        "Số phiên bản": cnt,
        "Tỷ lệ / 41.498 phiên bản (%)": 100.0 * cnt / scope["versions"],
        "Tỷ lệ / 38.554 bản TeX (%)": (100.0 * cnt / tex_total) if in_tex else np.nan,
    })
st_df = pd.DataFrame(rows_st)
display(style_table(
    st_df,
    caption="Bảng M2.1. Phân bố trạng thái phân tích trên 41.498 phiên bản và trên 38.554 phiên bản có mã nguồn TeX (97,34% dựng được cây).",
    int_cols=["Số phiên bản"],
    pct_cols=["Tỷ lệ / 41.498 phiên bản (%)", "Tỷ lệ / 38.554 bản TeX (%)"],
))

pm = pd.DataFrame(m2["per_member"])
view_m2 = pd.DataFrame({
    "Thành viên": pm["member"],
    "Phiên bản": pm["versions"],
    "Có TeX": pm["tex_verified"],
    "Success": pm["success"],
    "Partial": pm["partial"],
    "Empty": pm["empty_no_tex"],
    "Lỗi": pm["error"],
    "Success / TeX": 100.0 * pm["tex_success_rate"],
    "Dựng cây / TeX": 100.0 * pm["tex_parsed_rate"],
    "Thời gian (s)": pm["runtime_sec"],
    "Peak RAM (MiB)": pm["peak_rss_mib"],
})
display(style_table(
    view_m2,
    caption="Bảng M2.2. Kết quả phân tích và thông số tài nguyên trên Google Colab CPU-only của 5 thành viên.",
    int_cols=["Phiên bản", "Có TeX", "Success", "Partial", "Empty", "Lỗi"],
    pct_cols=["Success / TeX", "Dựng cây / TeX"],
))

dd_rows = []
for k, name_vi in [("elements", "Phần tử cấu trúc (elements)"), ("bibtex", "Mục tài liệu tham khảo (bibtex)")]:
    info = m2["deduplication"][k]
    dd_rows.append({
        "Bảng dữ liệu": name_vi,
        "Trước khử trùng": int(info["before"]),
        "Sau khử trùng (duy nhất)": int(info["after"]),
        "Số bản sao loại bỏ": int(info["before"] - info["after"]),
        "Tỷ lệ giảm (%)": 100.0 * float(info["reduction_rate"]),
    })
display(style_table(
    pd.DataFrame(dd_rows),
    caption="Bảng M2.3. Hiệu quả khử trùng hai tầng xuyên phiên bản trên 25.000 bài báo.",
    int_cols=["Trước khử trùng", "Sau khử trùng (duy nhất)", "Số bản sao loại bỏ"],
    pct_cols=["Tỷ lệ giảm (%)"],
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.8), gridspec_kw={"width_ratios": [1.18, 1.0]})
y1 = np.arange(len(st_df))
c1 = ["#1B365D", "#3B6E8C", "#94A3B8", "#64748B", "#CBD5E1"]
ax1.barh(y1, st_df["Số phiên bản"].to_numpy(dtype=float), color=c1, height=0.55)
ax1.set_yticks(y1)
ax1.set_yticklabels(["Success", "Partial", "Chỉ PDF (empty)", "Lỗi cú pháp", "Thiếu nguồn M1"])
ax1.invert_yaxis()
ax1.set_xlim(0, 44000)
ax1.grid(axis="x")
ax1.set_xlabel("Số phiên bản")
ax1.set_title("Hình M2.1a. Kết quả trên 41.498 phiên bản")
for idx_r, r in enumerate(st_df.to_dict("records")):
    ax1.text(float(r["Số phiên bản"]) + 450, idx_r,
             f"{fmt_int(r['Số phiên bản'])} ({fmt_pct(r['Tỷ lệ / 41.498 phiên bản (%)'])})",
             va="center", fontsize=8.0)

y2 = np.arange(len(pm))
tot_v = pm["versions"].to_numpy(dtype=float)
s_p = 100.0 * pm["success"].to_numpy(dtype=float) / tot_v
p_p = 100.0 * pm["partial"].to_numpy(dtype=float) / tot_v
e_p = 100.0 * pm["empty_no_tex"].to_numpy(dtype=float) / tot_v
err_p = 100.0 * (pm["error"].to_numpy(dtype=float) + pm["error_missing_source"].to_numpy(dtype=float)) / tot_v
ax2.barh(y2, s_p, color="#1B365D", height=0.54, label="Success")
ax2.barh(y2, p_p, left=s_p, color="#3B6E8C", height=0.54, label="Partial")
ax2.barh(y2, e_p, left=s_p + p_p, color="#94A3B8", height=0.54, label="Empty")
ax2.barh(y2, err_p, left=s_p + p_p + e_p, color="#CBD5E1", height=0.54, label="Lỗi / 404")
ax2.set_yticks(y2)
ax2.set_yticklabels([str(v) for v in pm["member"]])
ax2.invert_yaxis()
ax2.set_xlim(0, 100)
ax2.set_xlabel("Tỷ trọng phiên bản (%)")
ax2.set_title("Hình M2.1b. Độ đồng đều giữa 5 phân vùng")
ax2.legend(ncol=4, fontsize=7.6, loc="lower center", bbox_to_anchor=(0.5, -0.28), frameon=False)
for i_m, sp in enumerate(s_p):
    ax2.text(sp / 2, i_m, fmt_pct(sp, 1), ha="center", va="center", color="white", fontsize=7.9, fontweight="bold")
plt.tight_layout()
plt.show()

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 2 - Hierarchical Parsing and Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/main.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/main.py).
- Mã nguồn triển khai: [`src/parser/statistics.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/statistics.py).
- Mã nguồn triển khai: [`tools/m2_colab_runner.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_colab_runner.py).
- Mã nguồn triển khai: [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).